In this assignment, I am using the CIFAR-10 dataset, which consists of 60,000 color images in 10 classes, with 6,000 images per class. The goal is to apply model compression and optimization techniques to the dataset to see the benefit interms of high letency and low inference to improve model performamce.

In [1]:
#install touchvision
%pip install torch torchvision matplotlib pandas

Note: you may need to restart the kernel to use updated packages.


In [1]:
#import the necessary library
import copy
import os
import time

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

import matplotlib.pyplot as plt
import pandas as pd

torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Training device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA GPU unavailable. Training will use the CPU.")

Training device: cpu
CUDA GPU unavailable. Training will use the CPU.


os checks the saved model’s file size. 
time is used to measures how long predictions take. 
copy Keeps a separate copy of the best model’s weights. 
torch Provides PyTorch tools and tensors. 


In [2]:
#Load and prepare CIFAR-10. since CUDA is unavailable we will use CPU
# CIFAR-10 has three colour channels: red, green, and blue.
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

evaluation_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Training images with augmentation.
train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

# Same original images without augmentation for validation.
validation_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=evaluation_transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=evaluation_transform
)

# Balanced split: 4,500 training and 500 validation images per class.
generator = torch.Generator().manual_seed(42)
targets = torch.tensor(train_dataset.targets)

train_indices = []
validation_indices = []

for class_id in range(10):
    class_indices = torch.where(targets == class_id)[0]
    shuffled = class_indices[
        torch.randperm(len(class_indices), generator=generator)
    ]

    train_indices.extend(shuffled[:4500].tolist())
    validation_indices.extend(shuffled[4500:].tolist())

train_subset = Subset(train_dataset, train_indices)
validation_subset = Subset(
    validation_dataset, validation_indices
)

train_loader = DataLoader(
    train_subset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

validation_loader = DataLoader(
    validation_subset,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

print("Training images:", len(train_subset))
print("Validation images:", len(validation_subset))
print("Test images:", len(test_dataset))
print("Classes:", test_dataset.classes)

Training images: 45000
Validation images: 5000
Test images: 10000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [3]:
#Build the baseline CNN
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # 32 × 32 → 16 × 16
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout(0.1),

            # 16 × 16 → 8 × 8
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout(0.2),

            # 8 × 8 → 4 × 4
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = Net().to(device)
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

print("Model device:", next(model.parameters()).device)

Model device: cpu


Convolutional layers learns from the images. 
Pooling reduces their dimensions
dropout helps reduce overfitting and to ensure that the model is trained well
final layer outputs 10 class scores.

In [4]:
#Define training and evaluation functions
def train_one_epoch(model, loader, optimizer, run_device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(run_device, non_blocking=True)
        labels = labels.to(run_device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100 * correct / total


def evaluate(model, loader, run_device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(run_device)
            labels = labels.to(run_device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, 100 * correct / total

In [5]:
#create and train the baseline
epochs = 30 #Trains for 30 epochs and saves the best validation checkpoint. The learning rate gradually decreases.

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs
)

history = {
    "train_loss": [],
    "validation_loss": [],
    "train_accuracy": [],
    "validation_accuracy": []
}

best_validation_accuracy = -1.0

for epoch in range(1, epochs + 1):
    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, optimizer, device
    )

    validation_loss, validation_accuracy = evaluate(
        model, validation_loader, device
    )

    history["train_loss"].append(train_loss)
    history["validation_loss"].append(validation_loss)
    history["train_accuracy"].append(train_accuracy)
    history["validation_accuracy"].append(validation_accuracy)

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        torch.save(model.state_dict(), "baseline_model.pth")

    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{epochs} | "
        f"Train loss: {train_loss:.4f} | "
        f"Train accuracy: {train_accuracy:.2f}% | "
        f"Validation loss: {validation_loss:.4f} | "
        f"Validation accuracy: {validation_accuracy:.2f}%"
    )

# Restore the checkpoint with the best validation accuracy.
model.load_state_dict(
    torch.load(
        "baseline_model.pth",
        map_location=device,
        weights_only=True
    )
)

Epoch 01/30 | Train loss: 1.5511 | Train accuracy: 42.27% | Validation loss: 1.4120 | Validation accuracy: 49.18%
Epoch 02/30 | Train loss: 1.1636 | Train accuracy: 58.13% | Validation loss: 1.1046 | Validation accuracy: 61.50%
Epoch 03/30 | Train loss: 1.0058 | Train accuracy: 64.69% | Validation loss: 0.8233 | Validation accuracy: 71.04%
Epoch 04/30 | Train loss: 0.9079 | Train accuracy: 68.18% | Validation loss: 0.8816 | Validation accuracy: 69.90%
Epoch 05/30 | Train loss: 0.8488 | Train accuracy: 70.66% | Validation loss: 0.7463 | Validation accuracy: 73.78%
Epoch 06/30 | Train loss: 0.7977 | Train accuracy: 72.41% | Validation loss: 0.6789 | Validation accuracy: 76.34%
Epoch 07/30 | Train loss: 0.7473 | Train accuracy: 74.34% | Validation loss: 0.6312 | Validation accuracy: 78.74%
Epoch 08/30 | Train loss: 0.7154 | Train accuracy: 75.30% | Validation loss: 0.6040 | Validation accuracy: 79.32%
Epoch 09/30 | Train loss: 0.6780 | Train accuracy: 76.95% | Validation loss: 0.5715 | Va

<All keys matched successfully>

In [6]:
#Measure baseline accuracy, size, and latency
cpu = torch.device("cpu")
torch.set_num_threads(1)

# Same 200 preprocessed test images for every latency measurement.
latency_images = torch.stack([
    test_dataset[i][0] for i in range(200)
])


def measure_latency(model, images, repeats=3):
    model.eval()

    with torch.inference_mode():
        # Warm-up calls.
        for _ in range(20):
            model(images[:1])

        start = time.perf_counter()

        for _ in range(repeats):
            for image in images:
                model(image.unsqueeze(0))

        elapsed = time.perf_counter() - start

    return elapsed * 1000 / (repeats * len(images))


baseline_cpu = copy.deepcopy(model).cpu().eval()

_, baseline_accuracy = evaluate(
    baseline_cpu, test_loader, cpu
)

baseline_size = os.path.getsize("baseline_model.pth") / 1e6
baseline_latency = measure_latency(
    baseline_cpu, latency_images
)

print(f"Baseline test accuracy: {baseline_accuracy:.2f}%")
print(f"Baseline size: {baseline_size:.2f} MB")
print(f"Baseline CPU latency: {baseline_latency:.3f} ms/image")

if baseline_accuracy < 75:
    print("The baseline has not met the required 75% test accuracy.")

Baseline test accuracy: 87.35%
Baseline size: 3.28 MB
Baseline CPU latency: 3.341 ms/image


In [7]:
#pruning
pruned_model = Net().to(device)

pruned_model.load_state_dict(
    torch.load(
        "baseline_model.pth",
        map_location=device,
        weights_only=True
    )
)

# Zero the smallest 50% of weights in each selected layer.
for module in pruned_model.modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        prune.l1_unstructured(
            module, name="weight", amount=0.5
        )

# Lower learning rate for fine-tuning.
pruned_optimizer = optim.Adam(
    pruned_model.parameters(),
    lr=0.0001,
    weight_decay=1e-4
)

best_pruned_accuracy = -1.0
best_pruned_state = None

for epoch in range(1, 6):
    train_loss, train_accuracy = train_one_epoch(
        pruned_model,
        train_loader,
        pruned_optimizer,
        device
    )

    validation_loss, validation_accuracy = evaluate(
        pruned_model, validation_loader, device
    )

    if validation_accuracy > best_pruned_accuracy:
        best_pruned_accuracy = validation_accuracy
        best_pruned_state = {
            key: value.detach().cpu().clone()
            for key, value in pruned_model.state_dict().items()
        }

    print(
        f"Fine-tuning epoch {epoch} | "
        f"Train accuracy: {train_accuracy:.2f}% | "
        f"Validation accuracy: {validation_accuracy:.2f}%"
    )

pruned_model.load_state_dict(best_pruned_state)

# Remove masks while retaining the zeroed weights.
for module in pruned_model.modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        prune.remove(module, "weight")

torch.save(pruned_model.state_dict(), "pruned_model.pth")

pruned_cpu = copy.deepcopy(pruned_model).cpu().eval()

_, pruned_accuracy = evaluate(
    pruned_cpu, test_loader, cpu
)

pruned_size = os.path.getsize("pruned_model.pth") / 1e6
pruned_latency = measure_latency(
    pruned_cpu, latency_images
)

total_weights = 0
zero_weights = 0

for module in pruned_cpu.modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        total_weights += module.weight.numel()
        zero_weights += (module.weight == 0).sum().item()

print(f"Pruned test accuracy: {pruned_accuracy:.2f}%")
print(f"Pruned size: {pruned_size:.2f} MB")
print(f"Pruned CPU latency: {pruned_latency:.3f} ms/image")
print(
    f"Selected-layer weight sparsity: "
    f"{100 * zero_weights / total_weights:.2f}%"
)

Fine-tuning epoch 1 | Train accuracy: 83.51% | Validation accuracy: 86.42%
Fine-tuning epoch 2 | Train accuracy: 85.57% | Validation accuracy: 86.64%
Fine-tuning epoch 3 | Train accuracy: 85.80% | Validation accuracy: 86.88%
Fine-tuning epoch 4 | Train accuracy: 86.26% | Validation accuracy: 87.06%
Fine-tuning epoch 5 | Train accuracy: 86.66% | Validation accuracy: 86.82%
Pruned test accuracy: 86.88%
Pruned size: 3.28 MB
Pruned CPU latency: 3.717 ms/image
Selected-layer weight sparsity: 50.00%


In [9]:
# Quantization
supported_engines = torch.backends.quantized.supported_engines

# Include onednn in the available backend choices.
for engine in ("x86", "fbgemm", "onednn", "qnnpack"):
    if engine in supported_engines:
        torch.backends.quantized.engine = engine
        break
else:
    raise RuntimeError(
        f"No suitable INT8 backend available: {supported_engines}"
    )

print("Quantization backend:", torch.backends.quantized.engine)

# Dynamic INT8 quantization runs on the CPU.
cpu = torch.device("cpu")

quantized_model = torch.ao.quantization.quantize_dynamic(
    copy.deepcopy(baseline_cpu).cpu().eval(),
    {nn.Linear},
    dtype=torch.qint8
)

quantized_model.eval()

torch.save(
    quantized_model.state_dict(),
    "quantized_model.pth"
)

_, quantized_accuracy = evaluate(
    quantized_model, test_loader, cpu
)

quantized_size = os.path.getsize("quantized_model.pth") / 1e6

quantized_latency = measure_latency(
    quantized_model, latency_images.cpu()
)

print(f"Quantized test accuracy: {quantized_accuracy:.2f}%")
print(f"Quantized size: {quantized_size:.2f} MB")
print(
    f"Quantized CPU latency: "
    f"{quantized_latency:.3f} ms/image"
)

Quantization backend: onednn


C:\Users\tebos\AppData\Local\Temp\ipykernel_32412\1220817788.py:19: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.ao.quantization.quantize_dynamic(
C:\Users\tebos\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\ao\nn\quantized\modules\utils.py:72: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantize

Quantized test accuracy: 87.33%
Quantized size: 1.70 MB
Quantized CPU latency: 5.687 ms/image


In [10]:
#Display final comparism
results = pd.DataFrame({
    "Model": ["Baseline", "Pruned", "Quantized"],
    "Test accuracy (%)": [
        baseline_accuracy,
        pruned_accuracy,
        quantized_accuracy
    ],
    "Size (MB)": [
        baseline_size,
        pruned_size,
        quantized_size
    ],
    "CPU latency (ms/image)": [
        baseline_latency,
        pruned_latency,
        quantized_latency
    ]
})

results["Size reduction (%)"] = (
    100 * (1 - results["Size (MB)"] / baseline_size)
)

results["Speedup"] = (
    baseline_latency / results["CPU latency (ms/image)"]
)

display(results.round(3))

,Model,Test accuracy (%),Size (MB),CPU latency (ms/image),Size reduction (%),Speedup
0,Baseline,87.35,3.279,3.341,0.000,1.000
1,Pruned,86.88,3.279,3.717,0.003,0.899
2,Quantized,87.33,1.700,5.687,48.151,0.587


The baseline CNN achieved 87.35% test accuracy, exceeding the required 75%. 
Pruning preserved most of the accuracy but did not meaningfully reduce the saved model size or improve inference speed. 
Quantization reduced the model size by approximately 48% with almost no accuracy loss, but increased CPU inference latency. 
These results show that compression can reduce storage requirements without necessarily making predictions faster.